# 4장 상관과 회귀 (R과 파이썬으로 하는 통계분석)

보건통계학 기초 6장의 예로 Pearson·Spearman·Kendall 상관계수, 단순회귀와 두 구간, 다중회귀와 범주형 변수, 교호작용, 스플라인과 VIF, 로그 결과 모형, 공분산분석을 R로 계산합니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요. 패키지가 없으면 먼저 install.packages(c("boot", "sandwich", "lmtest", "multcomp", "car", "emmeans"))를 실행합니다.


## a. Pearson 상관계수와 신뢰구간

### 셀 1. 환자 8명의 Pearson 상관계수

In [ ]:
age <- c(42, 48, 53, 58, 62, 66, 71, 80)            # 나이 (세)
sbp <- c(125, 127, 133, 141, 129, 139, 137, 149)    # 수축기혈압 (mmHg)
cor.test(age, sbp)                                  # Pearson r, t, 양측 p, Fisher z 95% CI

### 셀 2. 결측값이 있으면

In [ ]:
sbp_na <- replace(sbp, 3, NA)                       # 셋째 환자의 혈압이 빠졌다면
cor(age, sbp_na)                                    # 결측이 있으면 NA
cor(age, sbp_na, use = "complete.obs")              # 두 값이 모두 있는 환자만
cor.test(age, sbp_na)$parameter                     # 자유도 5 = 분석한 쌍 7 − 2

### 셀 3. 부트스트랩 신뢰구간

In [ ]:
suppressPackageStartupMessages(library(boot))
set.seed(2026)
d8 <- data.frame(age, sbp)
b <- boot(d8, function(d, i) cor(d$age[i], d$sbp[i]), R = 4000)   # 환자를 짝지은 채 다시 뽑기
boot.ci(b, type = c("perc", "bca"))                 # 백분위수 구간과 BCa 구간

## b. Spearman과 Kendall 상관계수

### 셀 4. Spearman 상관계수와 두 가지 p

In [ ]:
cor.test(age, sbp, method = "spearman")                  # n < 1290이고 동점이 없으면 정확 p
cor.test(age, sbp, method = "spearman", exact = FALSE)   # t 근사 p

### 셀 5. Kendall의 τ

In [ ]:
cor.test(age, sbp, method = "kendall")              # 동점이 없고 n < 50이면 정확 p

### 셀 6. 극단값 한 명이 두 계수에 주는 영향

In [ ]:
nrs   <- c(2, 4, 5, 7, 8, 9)                        # 통증 점수
doses <- c(1, 0, 3, 2, 5, 4)                        # 추가 복용 횟수
doses50 <- replace(doses, 5, 50)                    # 한 명이 50회였다면
round(c(pearson = cor(nrs, doses), pearson50 = cor(nrs, doses50),
        spearman = cor(nrs, doses, method = "spearman"),
        spearman50 = cor(nrs, doses50, method = "spearman")), 3)

## c. 단순회귀와 두 구간

### 셀 7. 지역약국 참여자 100명의 단순회귀

In [ ]:
bp <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ch06_bp100.csv")
fit <- lm(sbp ~ age, data = bp)
summary(fit)
confint(fit)

### 셀 8. 평균의 신뢰구간과 개인의 예측구간

In [ ]:
new <- data.frame(age = 70)
predict(fit, new, interval = "confidence")          # 70세 평균의 95% 신뢰구간
predict(fit, new, interval = "prediction")          # 70세 한 사람의 95% 예측구간

### 셀 9. 잔차 그림

In [ ]:
par(mfrow = c(1, 2))
plot(fit, which = 1)                                # 잔차 대 예측값: 휘어짐, 부채꼴
plot(fit, which = 2)                                # 잔차의 Q-Q plot

### 셀 10. 이분산에 강건한 표준오차

In [ ]:
suppressPackageStartupMessages({library(sandwich); library(lmtest)})
coeftest(fit, vcov = vcovHC(fit, type = "HC3"))     # 이분산에 강건한 표준오차 (계수는 그대로)

### 셀 11. 출력의 요약값만으로 두 구간 다시 계산하기

In [ ]:
n <- 100; b0 <- 97.2326; b1 <- 0.5296; se_b1 <- 0.1169; s <- 14.5008   # 출력의 요약값
tc <- qt(0.975, n - 2)
round(c(t_crit = tc, lo = b1 - tc * se_b1, hi = b1 + tc * se_b1), 3)    # 기울기의 95% CI
sxx <- (s / se_b1)^2                                # SE(b1) = s / sqrt(Sxx)를 거꾸로 풂
yhat <- b0 + b1 * 70
se_m <- s * sqrt(1 / n + (70 - 56)^2 / sxx)         # 70세 평균의 표준오차 (평균 나이 56세)
se_p <- s * sqrt(1 + 1 / n + (70 - 56)^2 / sxx)     # 70세 한 사람 예측의 표준오차
round(c(yhat = yhat, se_mean = se_m, se_pred = se_p), 2)
round(c(yhat - tc * se_m, yhat + tc * se_m, yhat - tc * se_p, yhat + tc * se_p), 1)

## d. 다중회귀와 범주형 변수

### 셀 12. 고혈압 코호트 240명: 보정 전과 기저 SBP 보정

In [ ]:
d <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ch06_htn240.csv")
d$inst <- factor(d$inst, levels = c("clinic", "hospital", "tertiary"))   # 첫 수준 = 기준범주
d$sbp0_10 <- d$sbp0 / 10                            # 기저 SBP, 10 mmHg 단위
d$age10 <- d$age / 10                               # 나이, 10세 단위
m1 <- lm(sbp6 ~ classes, data = d)                  # 보정 전
m2 <- lm(sbp6 ~ classes + sbp0, data = d)           # 기저 SBP만 보정
rbind(crude = coef(summary(m1))["classes", ], base_adjusted = coef(summary(m2))["classes", ])

### 셀 13. 전체 보정 모형

In [ ]:
full <- lm(sbp6 ~ classes + sbp0_10 + age10 + female + diabetes + bmi + adherent + inst, data = d)
summary(full)
confint(full)["classes", ]
nobs(full)                                          # 분석에 들어간 환자 수 (결측이 있으면 줄어듦)

### 셀 14. 범주형 변수 전체의 F 검정

In [ ]:
red <- update(full, . ~ . - inst)                   # 요양기관 종별을 뺀 모형
anova(red, full)                                    # 가변수 2개를 한꺼번에: P for type of institution

### 셀 15. 기준범주와 숫자 코드

In [ ]:
d$inst_h <- relevel(d$inst, ref = "hospital")       # 기준범주를 병원으로 바꾸면
coef(update(full, . ~ . - inst + inst_h))[c("inst_hclinic", "inst_htertiary")]
d$inst_code <- as.integer(d$inst)                   # 1, 2, 3 코드를 factor 없이 넣으면
coef(update(full, . ~ . - inst + inst_code))["inst_code"]   # 연속형으로 계수 하나

### 셀 16. 교호작용

In [ ]:
fi <- lm(sbp6 ~ classes * diabetes + sbp0_10 + age10 + female + bmi + adherent + inst, data = d)
coef(summary(fi))[c("classes", "diabetes", "classes:diabetes"), ]
suppressPackageStartupMessages(library(multcomp))
confint(glht(fi, linfct = "classes + `classes:diabetes` = 0"))$confint[1, ]   # 당뇨 환자의 기울기와 95% CI
fi2 <- lm(sbp6 ~ classes * inst + sbp0_10 + age10 + female + diabetes + bmi + adherent, data = d)
anova(full, fi2)                                    # 계열 수 × 요양기관 종별 (자유도 2)

## e. 모형 점검과 로그 변환

### 셀 17. 직선이 맞는지: 스플라인과 범주

In [ ]:
library(splines)
sp <- lm(sbp6 ~ classes + ns(sbp0_10, df = 3) + age10 + female + diabetes + bmi + adherent + inst, data = d)
anova(full, sp)                                     # 기저 SBP: 곡선 대 직선
coef(sp)["classes"]
fc <- lm(sbp6 ~ factor(classes) + sbp0_10 + age10 + female + diabetes + bmi + adherent + inst, data = d)
coef(summary(fc))[2:4, ]                            # 1계열과 비교한 2·3·4계열
confint(fc)["factor(classes)4", ]
anova(full, fc)                                     # 계열 수: 범주 대 직선

### 셀 18. VIF와 설명변수의 겹침

In [ ]:
suppressPackageStartupMessages(library(car))
round(vif(full), 2)                                 # 범주형 inst가 있어 GVIF 열로 나옴
fw <- update(full, . ~ . + weight)                  # BMI와 겹치는 체중을 더 넣으면
round(cor(d$bmi, d$weight), 2)
coef(summary(fw))[c("bmi", "classes"), ]
round(vif(fw)[c("bmi", "weight"), "GVIF"], 2)

### 셀 19. 표준화 계수

In [ ]:
b <- coef(full)
round(c(classes = b[["classes"]] * sd(d$classes), sbp0 = b[["sbp0_10"]] * sd(d$sbp0_10),
        adherent = b[["adherent"]] * sd(d$adherent)) / sd(d$sbp6), 2)   # 표준화 계수

### 셀 20. 로그 의료비 모형

In [ ]:
cost <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ch06_cost400.csv")
fl <- lm(log(cost) ~ adherent + I(age / 10) + cci, data = cost)
coef(summary(fl))
round(100 * (exp(cbind(estimate = coef(fl), confint(fl))[-1, ]) - 1), 1)   # 백분율 차이(%)

## f. 공분산분석

### 셀 21. HbA1c 시험 72명의 공분산분석

In [ ]:
h <- read.csv("https://socialp-ajou.tecentriq12.workers.dev/data/ch06_hba1c72.csv")
anc <- lm(hba1c_6m ~ arm + hba1c_0, data = h)      # arm: 복약상담 1, 일반 진료 0
summary(anc)
confint(anc)["arm", ]

### 셀 22. 세 분석의 군 차이

In [ ]:
rbind(post   = coef(summary(lm(hba1c_6m ~ arm, data = h)))["arm", ],
      change = coef(summary(lm(I(hba1c_6m - hba1c_0) ~ arm, data = h)))["arm", ],
      ancova = coef(summary(anc))["arm", ])

### 셀 23. 군별 보정 평균과 기울기 점검

In [ ]:
suppressPackageStartupMessages(library(emmeans))
h$arm_f <- factor(h$arm, labels = c("usual", "counsel"))
emmeans(lm(hba1c_6m ~ arm_f + hba1c_0, data = h), ~ arm_f)   # 기저값을 전체 평균에 둔 군별 평균
anova(anc, lm(hba1c_6m ~ arm * hba1c_0, data = h))          # 두 군의 기울기가 같은가

## g. 과제 정답

### 셀 24. 과제 1 정답

In [ ]:
r <- 0.45; n <- 28
t <- r * sqrt(n - 2) / sqrt(1 - r^2)
c(t = t, p = 2 * pt(-abs(t), n - 2))
tanh(atanh(r) + c(-1, 1) * qnorm(0.975) / sqrt(n - 3))     # Fisher z 95% CI

### 셀 25. 과제 2 정답

In [ ]:
predict(fit, data.frame(age = c(40, 90)), interval = "confidence")
predict(fit, data.frame(age = c(40, 90)), interval = "prediction")
range(bp$age)

### 셀 26. 과제 3 정답

In [ ]:
fem <- lm(sbp6 ~ classes + sbp0_10 + age10 + diabetes + bmi + adherent + inst, data = subset(d, female == 1))
mal <- lm(sbp6 ~ classes + sbp0_10 + age10 + diabetes + bmi + adherent + inst, data = subset(d, female == 0))
rbind(women = coef(summary(fem))["classes", ], men = coef(summary(mal))["classes", ])
fx <- lm(sbp6 ~ classes * female + sbp0_10 + age10 + diabetes + bmi + adherent + inst, data = d)
coef(summary(fx))["classes:female", ]